# 034 — Optimización por enjambre y colonia

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** `v' = 0.6·2 + 1.5·0.4·(4−5) + 1.5·0.8·(1−5) = 1.2 − 0.6 − 4.8 = −4.2`. `x' = 5 − 4.2 = 0.8`. El término social domina (gbest lejos y r₂ alto) y lanza la partícula más allá del óptimo (x*=0): el sobretiro es normal en PSO.

**E2.** `f(0.8) = 0.64 < f(4) = 16` → **pbest ← 0.8**. `0.64 < f(gbest) = 1` → **gbest ← 0.8** (¡la partícula encontró el mejor global!). Paso 2: `v'' = 0.6·(−4.2) + 1.5·0.5·(0.8−0.8) + 1.5·0.5·(0.8−0.8) = −2.52` → `x'' = −1.72`. Con pbest = gbest = x, solo la inercia mueve: vuelve a sobrepasar y luego oscilará amortiguándose (si `w < 1`).

**E3.** Depósitos: `Δτ_A = 2·(1/2) = 1.0`, `Δτ_B = 1·(1/4) = 0.25`. Actualización: `τ_A = 0.5·1 + 1.0 = 1.5`; `τ_B = 0.5·1 + 0.25 = 0.75`. `P(A) = 1.5/2.25 = 2/3`. La ruta corta acumula ventaja, pero B conserva 1/3: la evaporación evita el monopolio inmediato.

**E4.** (1) En PSO la información compartida es continua y permanente (gbest visible para todos); en GA se comparte solo vía cruce entre pares. (2) La novedad en PSO viene de la inercia y los sorteos r₁,r₂; en GA, de mutación y recombinación. ACO es preferible en problemas combinatorios sobre grafos (rutas, TSP, scheduling) donde "posición + velocidad" continuas no tienen sentido natural.


In [ ]:
result = run_lab("optimization", seed=34)
assert result["kind"] == "optimization"
assert result["evidence"]
show(result)


In [ ]:
w, c1, c2 = 0.6, 1.5, 1.5
f = lambda x: x*x
x, v, pbest, gbest = 5.0, 2.0, 4.0, 1.0
v = w*v + c1*0.4*(pbest-x) + c2*0.8*(gbest-x)
x = x + v
print(f"E1: v'={v:.2f} x'={x:.2f}  f(x')={f(x):.2f}")
pbest = x if f(x) < f(pbest) else pbest
gbest = x if f(x) < f(gbest) else gbest
v = w*v + c1*0.5*(pbest-x) + c2*0.5*(gbest-x)
x = x + v
print(f"E2: v''={v:.2f} x''={x:.2f}")
rho = 0.5
tauA = (1-rho)*1 + 2*(1/2)
tauB = (1-rho)*1 + 1*(1/4)
print(f"E3: tauA={tauA} tauB={tauB} P(A)={tauA/(tauA+tauB):.3f}")


## Reflexión

1. En PSO, ¿qué comportamiento colectivo produce `c₂ ≫ c₁`? ¿Y el inverso? Conecta cada régimen con la convergencia prematura o la deriva.
2. ¿Por qué la evaporación de feromona en ACO cumple el mismo papel que la mutación en un GA? ¿Qué pasa con ρ = 0?
3. El laboratorio muestra mejora entre iteraciones. ¿Qué comparación mínima haría falta para afirmar que el enjambre supera a una búsqueda aleatoria con el mismo presupuesto de evaluaciones?
